# 01b · Pass 3 data: alternative sentiment labels and masked test sets

This notebook makes the extra files that Pass 3 needs. It does **not** change `gainsplit-processed`; you download its outputs and add them to that dataset as a **new version**, next to the existing files.

| Output | Used for |
|---|---|
| `helper_sent_alt.csv` | **RQ2 (helper-label quality).** Two other sentiment labelers (VADER, and a second pretrained model) for every headline, so we can swap the helper labels and see whether the gain depends on the teacher. |
| `cue_words.json` | **RQ3 (outlet-style shortcut).** The 100 words per class that a plain word-count model finds most telling in the training set. |
| `headlines_test_cuemask.csv` | **RQ3.** The test set with those cue words replaced by `[MASK]`. |
| `headlines_test_randmask.csv` | **RQ3 control.** The test set with the *same number* of words masked per headline, but chosen at random. |
| `manifest_pass3.json` | Fingerprints (SHA-256) of the four files, settings and package versions. |

**Before you press "Run All":**
1. **Settings → Internet → On** (downloads the model from Hugging Face, the code from GitHub, and `vaderSentiment` from pip).
2. **Add Input → `gainsplit-processed`.**
3. **Accelerator: GPU** is optional. It only makes step 4 faster.
4. **Open the model card of `TEACHER2_MODEL`** (link in step 4) *before* running and confirm it has 3 classes and was not trained only on tweets. If it does not, switch to the fallback constant.

## 1 · Get the code and settings

Clones the public repo into `/tmp` (like `02_train.ipynb`) so the masking functions come from the tested module `src/gainsplit/masking.py`, then prints the **commit** so every output can be traced to the exact code. The repo's masking tests are run first; if they fail, stop.

Everything you might want to change is in the settings cell; nothing below needs editing.

In [ ]:
import os, subprocess, sys, json, platform
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_masking.py", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-500:])
assert r.returncode == 0, "masking tests failed: the repo clone has no masking.py yet, or it is broken"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vaderSentiment"], check=True)

In [ ]:
import numpy as np
import pandas as pd
import sklearn, torch, transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from gainsplit.data import find_data_dir, verify, sha256
from gainsplit.masking import fit_cue_words, cue_mask, random_mask

pd.set_option("display.max_colwidth", 120)

# --- settings -------------------------------------------------------------
TEACHER2_MODEL          = "j-hartmann/sentiment-roberta-large-english-3-classes"   # check its model card first (step 4)
TEACHER2_FALLBACK_MODEL = "finiteautomata/bertweet-base-sentiment-analysis"        # use this if the check fails
MAX_LEN        = 48       # same as the existing teacher step in 01_preprocess.ipynb
TEACHER_BATCH  = 128
VADER_POS_THR  = 0.05     # compound >= 0.05 positive, <= -0.05 negative, else neutral
VADER_NEG_THR  = -0.05
K_PER_CLASS    = 100      # cue words kept per class
MASK_SEED      = 20261110 # random-mask control
LR_C, LR_SEED  = 1, 0     # cue-word model: LogisticRegression(C=1), random_state 0
OUT_DIR = Path("/kaggle/working/pass3")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SENT_CLASSES = ["negative", "neutral", "positive"]

print("python", platform.python_version(), "| pandas", pd.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("cuda:", torch.cuda.is_available())

## 2 · Load and verify the processed data

Finds `gainsplit-processed` under `/kaggle/input` and checks every file against `manifest.json`. This proves we label and mask exactly the headlines the training runs will use. The three splits are stacked into one table (with a `split` column); ids must match `helper_labels.csv`, because the new labels are joined to the old ones by id.

In [ ]:
DATA_DIR = find_data_dir()
manifest = verify(DATA_DIR)
print("data dir:", DATA_DIR, "| files verified:", len(manifest["outputs"]))

train = pd.read_csv(DATA_DIR / "headlines_train.csv")
dev   = pd.read_csv(DATA_DIR / "headlines_dev.csv")
test  = pd.read_csv(DATA_DIR / "headlines_test.csv")
helper = pd.read_csv(DATA_DIR / "helper_labels.csv")

all_hl = pd.concat([train.assign(split="train"), dev.assign(split="dev"), test.assign(split="test")], ignore_index=True)
all_hl["text"] = all_hl["text"].fillna("").astype(str)
assert all_hl["id"].is_unique
assert set(all_hl["id"]) == set(helper["id"]), "ids differ from helper_labels.csv"
print({k: len(v) for k, v in {"train": train, "dev": dev, "test": test}.items()}, "| total", len(all_hl))

## 3 · VADER sentiment (RQ2)

VADER is a **rule-based** sentiment scorer (a word list with scores, no neural network). Its `compound` score runs from -1 to +1; we call it positive at ≥ 0.05, negative at ≤ -0.05, otherwise neutral (the thresholds suggested by its authors). Why it matters: it has no connection to the sentiment teacher used so far, so if the helper-task gain survives when the labels come from VADER, the gain is not tied to one teacher's quirks.

In [ ]:
vader = SentimentIntensityAnalyzer()
all_hl["vader_compound"] = [vader.polarity_scores(t)["compound"] for t in all_hl["text"]]
all_hl["vader_label"] = np.where(all_hl["vader_compound"] >= VADER_POS_THR, "positive",
                         np.where(all_hl["vader_compound"] <= VADER_NEG_THR, "negative", "neutral"))
print(all_hl["vader_label"].value_counts())

## 4 · Second teacher model (RQ2)

A second neural sentiment model, different from `cardiffnlp/twitter-roberta-base-sentiment` (which was trained on tweets). If two teachers disagree on many headlines, the helper-task label is partly noise, which is exactly what RQ2 probes.

**Do this first:** open the model card, <https://huggingface.co/j-hartmann/sentiment-roberta-large-english-3-classes>, and confirm it (a) has **3 classes** and (b) was **not trained only on tweets**. If either fails, change `TEACHER2_MODEL` in step 1 to the fallback `finiteautomata/bertweet-base-sentiment-analysis` (note that one *is* tweet-based) and say so in the paper.

The class order is **read from the model's own `id2label`** and mapped by name to negative/neutral/positive; the cell stops with an error if the names are not recognised, rather than guessing.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NAME_MAP = {"negative": "negative", "neg": "negative", "neutral": "neutral", "neu": "neutral",
            "positive": "positive", "pos": "positive"}

tok = AutoTokenizer.from_pretrained(TEACHER2_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(TEACHER2_MODEL).to(DEVICE).eval()
id2label = dict(model.config.id2label)
print("model:", TEACHER2_MODEL, "| id2label:", id2label)
assert model.config.num_labels == 3, f"expected 3 classes, got {model.config.num_labels}"
col_of = {}   # class name -> column index in the model output
for i, name in id2label.items():
    key = NAME_MAP.get(str(name).strip().lower())
    if key is None:
        raise ValueError(f"cannot map label name {name!r} to negative/neutral/positive; check the model card")
    col_of[key] = int(i)
assert sorted(col_of) == SENT_CLASSES, f"labels map to {sorted(col_of)}, need each of {SENT_CLASSES} once"

texts = all_hl["text"].tolist()
chunks = []
with torch.no_grad():
    for i in range(0, len(texts), TEACHER_BATCH):
        enc = tok(texts[i:i + TEACHER_BATCH], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        chunks.append(torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy())
probs = np.concatenate(chunks)
assert probs.shape == (len(texts), 3)
del model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

for c in SENT_CLASSES:
    all_hl[f"teacher2_p_{c}"] = probs[:, col_of[c]].round(5)
all_hl["teacher2_label"] = np.array(SENT_CLASSES)[np.argmax(all_hl[[f"teacher2_p_{c}" for c in SENT_CLASSES]].to_numpy(), axis=1)]
print(all_hl["teacher2_label"].value_counts())

**Spot check:** read the most confident headlines of each class. If "positive" ones read as negative, the class mapping is wrong: stop and tell me.

In [ ]:
for c in SENT_CLASSES:
    top = all_hl.sort_values(f"teacher2_p_{c}", ascending=False).head(5)
    print(f"--- most {c} (teacher 2) ---")
    for t in top["text"]:
        print("  ", t)

## 5 · Cue words from the training set (RQ3)

The Onion and HuffPost write in different styles, so a model can score well by spotting outlet-style words instead of sarcasm. To measure how much, we fit a simple **word-count logistic regression** on the **full training set only** (never dev or test), then take the 100 words that push most strongly toward *sarcastic* and the 100 that push most strongly toward *non-sarcastic*. Words seen in fewer than 2 headlines are ignored. Tokens are whitespace-separated, the same unit the masking uses.

In [ ]:
cues = fit_cue_words(train, K_PER_CLASS)
cue_words = [w for w, _ in cues["sarcastic"]] + [w for w, _ in cues["non_sarcastic"]]
assert len(cue_words) == len(set(cue_words)) == 2 * K_PER_CLASS
print("top sarcastic:", [w for w, _ in cues["sarcastic"][:10]])
print("top non-sarcastic:", [w for w, _ in cues["non_sarcastic"][:10]])

## 6 · Masked test sets (RQ3)

- **Cue mask:** every word in the 200-word list becomes `[MASK]`. If a model's score falls a lot, it was leaning on those words.
- **Random mask (control):** in each headline, exactly as many words are masked as the cue mask masked there, but at random positions. This separates "the model lost the cue words" from "the model lost some words". Headlines with no cue word stay unchanged in both files.

Ids, row order and labels are the same as `headlines_test.csv` (checked below).

In [ ]:
test_texts = test["text"].fillna("").astype(str).tolist()
cm_texts, counts = cue_mask(test_texts, cue_words)
rm_texts = random_mask(test_texts, counts, MASK_SEED)

cuemask = test.copy();  cuemask["text"] = cm_texts
randmask = test.copy(); randmask["text"] = rm_texts
for df in (cuemask, randmask):
    assert df["id"].tolist() == test["id"].tolist() and df["label"].tolist() == test["label"].tolist()
assert [t.split().count("[MASK]") for t in rm_texts] == [t.split().count("[MASK]") for t in cm_texts] == counts

counts = np.array(counts)
coverage = {"test_headlines": int(len(counts)),
            "share_with_at_least_one_masked_word": float((counts >= 1).mean()),
            "mean_masked_words_per_headline": float(counts.mean()),
            "mean_masked_words_per_affected_headline": float(counts[counts >= 1].mean()) if (counts >= 1).any() else 0.0,
            "mean_tokens_per_test_headline": float(np.mean([len(t.split()) for t in test_texts]))}
print(coverage)

## 7 · Save the files and the manifest

Writes the four output files and `manifest_pass3.json` with each file's size and SHA-256 (same hashing as `manifest.json`), plus the settings and package versions. This is what lets the paper say exactly which labels and masks were used.

In [ ]:
alt = all_hl[["id", "split", "vader_label", "vader_compound", "teacher2_label"] + [f"teacher2_p_{c}" for c in SENT_CLASSES]]
alt = alt.sort_values("id").reset_index(drop=True)
alt["vader_compound"] = alt["vader_compound"].round(4)
assert alt["id"].tolist() == sorted(helper["id"].tolist())
alt.to_csv(OUT_DIR / "helper_sent_alt.csv", index=False)

(OUT_DIR / "cue_words.json").write_text(json.dumps(
    {"k_per_class": K_PER_CLASS, "sarcastic": cues["sarcastic"], "non_sarcastic": cues["non_sarcastic"], "coverage": coverage},
    indent=2, ensure_ascii=False))
cuemask.to_csv(OUT_DIR / "headlines_test_cuemask.csv", index=False)
randmask.to_csv(OUT_DIR / "headlines_test_randmask.csv", index=False)

names = ["helper_sent_alt.csv", "cue_words.json", "headlines_test_cuemask.csv", "headlines_test_randmask.csv"]
def pkg(n):
    try: return metadata.version(n)
    except Exception: return None
manifest_p3 = {
    "outputs": {n: {"bytes": (OUT_DIR / n).stat().st_size, "sha256": sha256(OUT_DIR / n)} for n in names},
    "settings": {"TEACHER2_MODEL": TEACHER2_MODEL, "TEACHER2_FALLBACK_MODEL": TEACHER2_FALLBACK_MODEL,
                 "MAX_LEN": MAX_LEN, "VADER_POS_THR": VADER_POS_THR, "VADER_NEG_THR": VADER_NEG_THR,
                 "K_PER_CLASS": K_PER_CLASS, "MASK_SEED": MASK_SEED, "LR_C": LR_C, "LR_SEED": LR_SEED,
                 "teacher2_id2label": {str(k): v for k, v in id2label.items()}},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                 "sklearn": sklearn.__version__, "torch": torch.__version__, "transformers": transformers.__version__,
                 "vaderSentiment": pkg("vaderSentiment"), "code_commit": COMMIT},
}
(OUT_DIR / "manifest_pass3.json").write_text(json.dumps(manifest_p3, indent=2))
for n, i in manifest_p3["outputs"].items():
    print(f"{n:32s} {i['bytes']:>10,d} bytes  {i['sha256'][:16]}...")

## 8 · Sanity checks (read these before uploading)

What to look for:
- **Coverage:** a clear share of test headlines should contain at least one cue word, with 1 to 3 words masked on average among those; if it is tiny (< 5%) the mask test has no power, if huge the headlines are mostly masked.
- **Cue words:** the sarcastic list should look Onion-like and the other HuffPost-like. If one side is dominated by junk (numbers, names of a single story), flag it.
- **Label counts:** all three classes should be present for both VADER and teacher 2.
- **Crosstabs:** agreement with the existing `sent_label` should be clearly above chance. VADER disagreeing a lot is expected; teacher 2 disagreeing a lot is worth noting in the paper.

In [ ]:
print("COVERAGE"); print(json.dumps(coverage, indent=2))
print("\nTOP 20 CUE WORDS")
print("sarcastic    :", [f"{w} ({v:.2f})" for w, v in cues["sarcastic"][:20]])
print("non-sarcastic:", [f"{w} ({v:.2f})" for w, v in cues["non_sarcastic"][:20]])

print("\nLABEL COUNTS PER SOURCE")
print(pd.DataFrame({"existing (sent_label)": helper["sent_label"].value_counts(),
                    "vader": alt["vader_label"].value_counts(),
                    "teacher2": alt["teacher2_label"].value_counts()}))

cmp = alt.merge(helper[["id", "sent_label"]], on="id", validate="one_to_one")
for col in ["vader_label", "teacher2_label"]:
    print(f"\n{col} (rows) vs existing sent_label (columns)")
    print(pd.crosstab(cmp[col], cmp["sent_label"], margins=True))
    print("agreement:", round(float((cmp[col] == cmp["sent_label"]).mean()), 3))